# 🧠 GRAN BATALLA DE INTELIGENCIAS ARTIFICIALES - LOTTO ACTIVO
### Comparativa de Modelos Avanzados de Machine Learning y Deep Learning

En este experimento avanzaremos al nivel de modelado predictivo más potente:
1. **Random Forest**: Nuestro modelo base con múltiples árboles de decisión.
2. **XGBoost (Gradient Boosting)**: Algoritmo secuencial que aprende y corrige sus errores celda por celda.
3. **LSTM (Long Short-Term Memory)**: Red Neuronal Recurrente capaz de recordar secuencias y tendencias de largo plazo (como los últimos 12 sorteos).
4. **Híbrido (LSTM + XGBoost)**: La red LSTM extrae los patrones temporales de fondo y se los pasa a XGBoost para que tome la decisión final de clasificación.
5. **Segmentación K-Means (Clustering)**: Agrupamos los animales en base a su comportamiento histórico (frecuencia de aparición y promedio de sorteos en espera) para ver si existen "familias" o ciclos estadísticos.

Ejecuta cada bloque en orden utilizando el botón de play ▶️.

In [ ]:
# CELDA 1: INSTALAR LIBRERIAS Y COMPONENTES
!pip install gspread google-auth pandas scikit-learn xgboost tensorflow matplotlib seaborn -q
print("\n" + "="*50)
print("CELDA 1 COMPLETADA - Entorno listo y librerías instaladas")
print("="*50)

In [ ]:
# CELDA 2: DESCARGAR HISTORIAL DE SORTEOS DE GOOGLE SHEETS
# IMPORTANTE: Sube tu archivo credentials.json a Colab usando la carpeta de la izquierda
import gspread
from google.oauth2.service_account import Credentials
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Conectando de forma segura con Google Sheets...")
scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
creds = Credentials.from_service_account_file('credentials.json', scopes=scope)
client = gspread.authorize(creds)

sheet = client.open("Lotto Activo - Resultados").worksheet("Resultados")
data = sheet.get_all_records()
df = pd.DataFrame(data)

print("\n" + "="*50)
print(f"CELDA 2 COMPLETADA - Descargados {len(df)} sorteos de Lotto Activo")
print("="*50)

In [ ]:
# CELDA 3: PREPARACIÓN DE CARACTERÍSTICAS Y SECUENCIAS TEMPORALES
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df['Datetime'] = pd.to_datetime(df['Fecha'] + ' ' + df['Hora'])
df = df.sort_values('Datetime').reset_index(drop=True)
df['Animalito'] = df['Animalito'].str.strip()

# Creación de Variables Temporales
df['Dia_Semana'] = df['Datetime'].dt.dayofweek
df['Hora_Num'] = df['Datetime'].dt.hour
df['Minuto'] = df['Datetime'].dt.minute
df['Es_Fin_De_Semana'] = df['Dia_Semana'].apply(lambda x: 1 if x >= 5 else 0)
df['Dia_Del_Mes'] = df['Datetime'].dt.day
df['Semana_Del_Anio'] = df['Datetime'].dt.isocalendar().week.astype(int)

# Variables de Retraso Temporal (Lag Features)
lags_a_usar = [1, 2, 3, 4, 5, 6, 12, 24]
for lag in lags_a_usar:
    df[f'Lag_{lag}'] = df['Animalito'].shift(lag)

df = df.dropna().copy()

# Convertimos categorías de texto (animales) a números identificadores únicos
le = LabelEncoder()
df['Target'] = le.fit_transform(df['Animalito'])
num_clases = len(le.classes_)

for lag in lags_a_usar:
    df[f'Lag_{lag}_Enc'] = le.transform(df[f'Lag_{lag}'])

# Consolidamos la lista de entradas para los modelos clásicos
features = [f'Lag_{l}_Enc' for l in lags_a_usar] + ['Dia_Semana', 'Hora_Num', 'Minuto', 'Es_Fin_De_Semana', 'Dia_Del_Mes', 'Semana_Del_Anio']
X = df[features].values
y = df['Target'].values

# Separación respetando orden cronológico sin mezclar el pasado con el futuro
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

azar_matematico = (1 / num_clases) * 100

print("\n" + "="*50)
print(f"CELDA 3 COMPLETADA - Datos listos para entrenamiento")
print(f"Total de animales diferentes registrados: {num_clases}")
print(f"Parámetros de entrada por sorteo: {len(features)}")
print(f"Datos de Entrenamiento: {len(X_train)} sorteos")
print(f"Datos de Validación/Testeo: {len(X_test)} sorteos")
print(f"Azar matemático de referencia (1/{num_clases}): {azar_matematico:.2f}%")
print("="*50)

In [ ]:
# CELDA 4: MODELO 1 - RANDOM FOREST (Bosque Aleatorio)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("Entrenando Random Forest con 500 árboles de decisión...")
rf = RandomForestClassifier(n_estimators=500, max_depth=20, min_samples_split=5, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred) * 100

print("\n" + "*"*55)
print(f"PRECISIÓN RANDOM FOREST: {rf_acc:.2f}%  (Azar esperado: {azar_matematico:.2f}%)")
print("*"*55)

In [ ]:
# CELDA 5: MODELO 2 - XGBOOST CLASSIFIER (Gradient Boosting)
from xgboost import XGBClassifier

print("Entrenando XGBoost (Corrección de errores iterativa)... Esto puede demorar un poco...")
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    num_class=num_clases,
    objective='multi:softmax',
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred) * 100

print("\n" + "*"*55)
print(f"PRECISIÓN XGBOOST: {xgb_acc:.2f}%  (Azar esperado: {azar_matematico:.2f}%)")
print("*"*55)

In [ ]:
# CELDA 6: MODELO 3 - DEEP LEARNING CON RED NEURONAL RECURRENTE (LSTM)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print("Estructurando Red Neuronal LSTM con memoria secuencial de 12 pasos...")

# Formateamos los datos secuencialmente usando ventanas de tiempo deslizantes
window_size = 12
secuencias_hist = df['Target'].values

X_seq, y_seq = [], []
for i in range(window_size, len(secuencias_hist)):
    X_seq.append(secuencias_hist[i-window_size:i])
    y_seq.append(secuencias_hist[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Ajustamos dimensiones para entrada de la capa LSTM [ejemplos, pasos_tiempo, variables]
X_seq = X_seq.reshape((X_seq.shape[0], X_seq.shape[1], 1))
y_seq_onehot = to_categorical(y_seq, num_classes=num_clases)

# Dividimos respetando el orden del tiempo
split_lstm = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:split_lstm], X_seq[split_lstm:]
y_train_seq, y_test_seq = y_seq_onehot[:split_lstm], y_seq_onehot[split_lstm:]
y_test_labels = y_seq[split_lstm:]

# Arquitectura Deep Learning usando API funcional para compatibilidad de entradas en Keras 3 (Colab)
inputs = Input(shape=(window_size, 1))
x = LSTM(128, return_sequences=True)(inputs)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = LSTM(64)(x)
x = Dropout(0.3)(x)
features_layer = Dense(64, activation='relu')(x)
outputs = Dense(num_clases, activation='softmax')(features_layer)

lstm_model = Model(inputs=inputs, outputs=outputs)
lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

print("Entrenando Red Neuronal en Colab... (1 - 3 minutos)")
history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=40,
    batch_size=64,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

lstm_pred = np.argmax(lstm_model.predict(X_test_seq), axis=1)
lstm_acc = accuracy_score(y_test_labels, lstm_pred) * 100

print("\n" + "*"*55)
print(f"PRECISIÓN RED NEURONAL LSTM: {lstm_acc:.2f}%  (Azar esperado: {azar_matematico:.2f}%)")
print("*"*55)


In [ ]:
# CELDA 7: MODELO 4 - MODELO HÍBRIDO (Extracción con LSTM + Clasificación XGBoost)
from tensorflow.keras.models import Model

print("Iniciando arquitectura Híbrida...")
print("Paso 1: Extrayendo representaciones abstractas y patrones con la capa intermedia del LSTM...")

# Usamos la capa de características funcionales directamente
extractor_capa = Model(inputs=lstm_model.input, outputs=lstm_model.layers[-2].output)
features_train_abstract = extractor_capa.predict(X_train_seq)
features_test_abstract = extractor_capa.predict(X_test_seq)

print("Paso 2: Entrenando XGBoost utilizando las características de la red neuronal...")
hibrido_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    num_class=num_clases,
    objective='multi:softmax',
    random_state=42,
    verbosity=0
)

y_train_seq_labels = y_seq[:split_lstm]
hibrido_xgb.fit(features_train_abstract, y_train_seq_labels)
hibrido_pred = hibrido_xgb.predict(features_test_abstract)
hibrido_acc = accuracy_score(y_test_labels, hibrido_pred) * 100

print("\n" + "*"*55)
print(f"PRECISIÓN MODELO HÍBRIDO (LSTM + XGB): {hibrido_acc:.2f}%  (Azar esperado: {azar_matematico:.2f}%)")
print("*"*55)


In [ ]:
# CELDA 8: SEGMENTACIÓN DE COMPORTAMIENTO (CLUSTERING K-MEANS)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

print("Calculando comportamiento estadístico de cada animal...")

# 1. Frecuencia de aparición
frecuencias = df['Animalito'].value_counts().to_dict()

# 2. Promedio y máximo tiempo en espera histórico sin salir
esperas_por_animal = {anim: [] for anim in df['Animalito'].unique()}
ultimo_visto = {anim: 0 for anim in df['Animalito'].unique()}

for idx, row in df.iterrows():
    anim = row['Animalito']
    if ultimo_visto[anim] != 0:
        esperas_por_animal[anim].append(idx - ultimo_visto[anim])
    ultimo_visto[anim] = idx

promedio_espera = {anim: (np.mean(lista) if lista else 0) for anim, lista in esperas_por_animal.items()}
max_espera = {anim: (np.max(lista) if lista else 0) for anim, lista in esperas_por_animal.items()}

# Crear DataFrame de Comportamiento
datos_cluster = []
for anim in frecuencias.keys():
    datos_cluster.append({
        'Animalito': anim,
        'Frecuencia': frecuencias[anim],
        'Espera_Promedio': promedio_espera[anim],
        'Espera_Maxima': max_espera[anim]
    })
df_cluster = pd.DataFrame(datos_cluster)

# Escalar los datos para que K-Means funcione correctamente
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[['Frecuencia', 'Espera_Promedio', 'Espera_Maxima']])

# Agrupamos en 3 Clústers (Familias: Salidores, Estables/Promedio, Lentos/Fríos)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

# Renombrar clusters de acuerdo a sus características medias
centros = df_cluster.groupby('Cluster')['Frecuencia'].mean().sort_values(ascending=False).index
nombres_clusters = {centros[0]: '🏆 Salidores frecuentes', centros[1]: '⚖️ Ciclo estable / Normal', centros[2]: '❄️ Lentos / Raros'}
df_cluster['Familia'] = df_cluster['Cluster'].map(nombres_clusters)

print("\n--- FAMILIAS DE ANIMALES IDENTIFICADAS POR CLUSTERING ---")
for familia in nombres_clusters.values():
    miembros = df_cluster[df_cluster['Familia'] == familia]['Animalito'].tolist()
    print(f"\n{familia} ({len(miembros)} animales):\n  {', '.join(miembros[:15])}... y más.")

# Visualización del Clustering
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_cluster,
    x='Frecuencia',
    y='Espera_Promedio',
    hue='Familia',
    palette='Set1',
    s=150,
    edgecolor='black'
)
for i, row in df_cluster.iterrows():
    plt.text(row['Frecuencia'] + 0.3, row['Espera_Promedio'], row['Animalito'], fontsize=9, alpha=0.8)

plt.title("Segmentación de Comportamiento K-Means (Familias de Ruleta)", fontsize=14, fontweight='bold')
plt.xlabel("Frecuencia Total de Apariciones")
plt.ylabel("Espera Promedio entre Apariciones (Sorteos)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("CELDA 8 COMPLETADA - Análisis de Clustering Terminado")
print("="*50)

In [ ]:
# CELDA 9: COMPARATIVA FINAL Y ANÁLISIS DE RESULTADOS
import matplotlib.pyplot as plt

modelos_nombres = [
    'Azar Puro\n(Referencia)',
    'Random Forest\n(Base)',
    'XGBoost\n(Corrector)',
    'LSTM\n(Red Neuronal)',
    'Híbrido\n(LSTM + XGB)'
]
valores_precision = [azar_matematico, rf_acc, xgb_acc, lstm_acc, hibrido_acc]
colores_grafico = ['#8A8A8A', '#4CAF50', '#FF9800', '#00BCD4', '#9C27B0']

print("\n" + "#"*60)
print("      TABLA COMPARATIVA DE PRECISIÓN DE INTELIGENCIAS ARTIFICIALES")
print("#"*60)
print(f"  Azar Matemático (1/{num_clases}):    {azar_matematico:.2f}%")
print(f"  Random Forest Classifier:    {rf_acc:.2f}%")
print(f"  XGBoost Classifier:          {xgb_acc:.2f}%")
print(f"  Red Neuronal Recurrente LSTM: {lstm_acc:.2f}%")
print(f"  Modelo Híbrido (LSTM+XGB):    {hibrido_acc:.2f}%")
print("#"*60 + "\n")

ganador_idx = np.argmax(valores_precision[1:]) + 1
nombre_ganador = modelos_nombres[ganador_idx].replace('\n', ' ')
precision_ganador = valores_precision[ganador_idx]
print(f"🏆 GANADOR DEL EXPERIMENTO: {nombre_ganador} con {precision_ganador:.2f}%")
print(f"   Diferencia sobre el azar teórico: {precision_ganador - azar_matematico:+.2f}%")

if precision_ganador > azar_matematico + 2.0:
    print("\n⚠️ ¡ATENCIÓN! La diferencia es estadísticamente notable. Existe un pequeño sesgo aprovechable o inestabilidad física detectada.")
else:
    print("\n✅ VEREDICTO DE CIENCIA DE DATOS: Lotto Activo opera de forma equilibrada y caótica.")
    print("   Las inteligencias artificiales no logran una ventaja significativa sobre el azar.")

# Gráfico
plt.figure(figsize=(12, 6))
bars = plt.bar(modelos_nombres, valores_precision, color=colores_grafico, edgecolor='black', linewidth=1.2)
plt.axhline(y=azar_matematico, color='red', linestyle='--', linewidth=2, label=f'Azar Puro ({azar_matematico:.2f}%)')

for bar, val in zip(bars, valores_precision):
    plt.text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.15,
        f'{val:.2f}%',
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=12
    )

plt.title('Gran Batalla de IAs en Lotto Activo vs Azar Teórico', fontsize=15, fontweight='bold')
plt.ylabel('Precisión (%)', fontsize=12)
plt.ylim(0, max(valores_precision) + 3)
plt.legend(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("CELDA 9 COMPLETADA - Experimento Finalizado")
print("="*50)